In [ ]:
!pip install -Uq vllm qdrant-client FlagEmbedding

In [ ]:
# Copy the database from the read-only input to the read-write working directory
!cp -r /kaggle/input/notebooks/shams03/hybridrag/medical_qdrant_db /kaggle/working/medical_qdrant_db

In [ ]:
import os
import time
from vllm import LLM, SamplingParams
from qdrant_client import QdrantClient
from qdrant_client import models
from FlagEmbedding import BGEM3FlagModel
from transformers import AutoTokenizer # <-- NEW: Required for Command R Native RAG
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 1. THE KAGGLE CUDA FIX
# ==========================================
nvidia_lib = "/usr/local/nvidia/lib64"
cuda_compat = "/usr/local/cuda-12.5/compat"
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"{nvidia_lib}:{cuda_compat}:{current_lib_path}"

# ==========================================
# 2. BOOT THE 8-BIT AWQ MODEL & TOKENIZER (vLLM)
# ==========================================
repo_id = "Shams03/ArEgy-Medical-Command-R7B-AWQ-8Bit"

print("🚀 Loading Tokenizer for Native RAG...")
tokenizer = AutoTokenizer.from_pretrained(repo_id)

print("🚀 Booting up vLLM Engine (8-Bit W8A16)...")
llm = LLM(
    model=repo_id,
    tensor_parallel_size=2, 
    max_model_len=2076,           
    gpu_memory_utilization=0.70,  # Crucial: Leaves VRAM for BGE-M3
    enforce_eager=True,           
)

sampling_params = SamplingParams(
    temperature=0.3, 
    max_tokens=500,
    repetition_penalty=1.00 # <-- Added to prevent reasoning loops in strict medical texts
    # Removed manual "stop" as the Chat Template handles native <EOS> tokens perfectly
)

# ==========================================
# 3. BOOT BAAI/bge-m3 EMBEDDING MODEL
# ==========================================
embedder = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda:1')

# ==========================================
# 4. CONNECT TO QDRANT HYBRID DB
# ==========================================
db_path = "/kaggle/working/medical_qdrant_db"
collection_name = "arabic_medical_HybridRAG"

print("\n🔍 Connecting to Local Qdrant Database...")
client = QdrantClient(path=db_path)

print("\n✅ System Initialization Complete. Ready for Inference!")

In [ ]:
import time
from qdrant_client import models

collection_name = "arabic_medical_HybridRAG"
def ask_medical_bot(user_question, system_preamble, top_k=3):
    start_time = time.time()
    
    # ==========================================
    # 1. GENERATE DENSE & SPARSE VECTORS (BGE-M3)
    # ==========================================
    embeddings = embedder.encode_queries([user_question], return_dense=True, return_sparse=True)
    
    dense_vec = embeddings['dense_vecs'][0].tolist()
    sparse_dict = embeddings['lexical_weights'][0]
    
    sparse_vec = models.SparseVector(
        indices=[int(k) for k in sparse_dict.keys()],
        values=list(sparse_dict.values())
    )

    # ==========================================
    # 2. RETRIEVE DOCUMENTS (Qdrant RRF)
    # ==========================================
    search_results = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(query=dense_vec, using="dense", limit=top_k * 2),
            models.Prefetch(query=sparse_vec, using="sparse", limit=top_k * 2),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k
    ).points

    # ==========================================
    # 3. FORMAT RETRIEVED DOCUMENTS AS DICTIONARIES (Command R Standard)
    # ==========================================
    docs_list = []
    for pt in search_results:
        source = pt.payload.get('source', pt.payload.get('Source', 'Unknown'))
        page = pt.payload.get('page', pt.payload.get('page_number', 'None'))
        text = pt.payload.get('text', pt.payload.get('page_content', pt.payload.get('content', '')))
        
        # Command R requires key-value pairs. "title" is used for citations.
        docs_list.append({
            "title": f"Source: {source} | Page: {page}",
            "text": text
        })

    # ==========================================
    # 4. CONSTRUCT NATIVE RAG PROMPT & GENERATE
    # ==========================================
    messages = [
        {"role": "system", "content": system_preamble},
        {"role": "user", "content": user_question}
    ]
    
    # This is the "Secret Sauce". It formats the prompt natively for Command R.
    full_prompt = tokenizer.apply_chat_template(
        conversation=messages,
        documents=docs_list, 
        tokenize=False,
        add_generation_prompt=True
    )
    
    outputs = llm.generate(prompts=[full_prompt], sampling_params=sampling_params, use_tqdm=False)
    generated_text = outputs[0].outputs[0].text.strip()
    
    latency = time.time() - start_time
    
    # ==========================================
    # 5. OUTPUT
    # ==========================================
    print("\n" + "="*70)
    print(f"👤 Patient: {user_question}")
    print("-" * 70)
    print(f"🤖 Medical Assistant (Latency: {latency:.2f}s):\n{generated_text}")
    print("="*70 + "\n")

In [ ]:
import time
import re
import urllib.parse

user_query = "كم عدد مرات رضاعة الطفل اللازمة في اليوم؟" # You can test a non-medical query here too!
top_k = 4 

# ==========================================
# 1. THE STRICT, MEDICAL-ONLY PREAMBLE
# ==========================================
system_preamble = """أنت مساعد طبي صارم ودقيق.
تعليماتك الأساسية:
1. الأسئلة غير الطبية: يُمنع تماماً الإجابة على أي سؤال خارج النطاق الطبي. إذا كان السؤال غير طبي، أجب حرفياً بهذه العبارة فقط: "عذراً، أنا مخصص للإجابة على الاستفسارات الطبية فقط."
2. صياغة المصدر الإجبارية: يجب أن تبدأ إجابتك حصراً بهذه الصيغة: "بناءً على [اسم المصدر]، صفحة [رقم الصفحة]: " (ثم اذكر الإجابة).
3. استخرج الإجابة من النصوص المرفقة فقط. تجاهل الكلمات غير المفهومة أو الأرقام المكررة الناتجة عن أخطاء القراءة الآلية (OCR)، واستخرج المعنى الطبي المفيد فقط.
4. إذا لم تجد الإجابة في النصوص، قل: "لا توجد إجابة دقيقة في النصوص المرفقة، يرجى استشارة طبيب مختص." ولا تؤلف إجابة من عندك."""

# ==========================================
# 2. EMBED & RETRIEVE
# ==========================================
start_time = time.time()
embeddings = embedder.encode_queries([user_query], return_dense=True, return_sparse=True)
sparse_dict = embeddings['lexical_weights'][0]
sparse_vec = models.SparseVector(
    indices=[int(k) for k in sparse_dict.keys()],
    values=list(sparse_dict.values())
)

search_results = client.query_points(
    collection_name=collection_name,
    prefetch=[
        models.Prefetch(query=embeddings['dense_vecs'][0].tolist(), using="dense", limit=top_k * 2),
        models.Prefetch(query=sparse_vec, using="sparse", limit=top_k * 2),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=top_k
).points

# ==========================================
# 3. HUMAN-LIKE DOCUMENT FORMATTING & DEBUG
# ==========================================
docs_str = ""
accepted_count = 0

print("\n" + "="*80)
print("🔍 DEBUG 1: RAW QDRANT RETRIEVAL (Score Analysis)")
print("-" * 80)

for i, pt in enumerate(search_results, 1):
    raw_source = pt.payload.get('source', pt.payload.get('Source', 'مصدر مجهول'))
    clean_source = urllib.parse.unquote(raw_source)
    
    print(f"Doc {i} | Score: {pt.score:.4f} | Source: {clean_source}")
    
    if pt.score <= 0.6:
        print(f"   -> ❌ REJECTED (Score <= 0.6)\n")
        continue
        
    print(f"   -> ✅ ACCEPTED\n")
    accepted_count += 1
    
    page = pt.payload.get('page', pt.payload.get('page_number', '1'))
    text = pt.payload.get('text', pt.payload.get('page_content', pt.payload.get('content', '')))
    
    if not str(page).isdigit():
        page = "1"
    
    docs_str += f"- المصدر: {clean_source} (صفحة {page})\n"
    docs_str += f"المعلومات: {text}\n\n"

# ==========================================
# 4. NATURAL PROMPT ASSEMBLY & TOKENIZATION
# ==========================================
if accepted_count > 0:
    user_content = f"إليك بعض النصوص الطبية المستخرجة:\n\n{docs_str}سؤال المريض: {user_query}\n\nأرجو الإجابة على السؤال باختصار وبناءً على هذه النصوص فقط."
else:
    user_content = f"سؤال المريض: {user_query}\n\nلم أجد أي نصوص طبية متعلقة بهذا السؤال للمساعدة."

messages = [
    {"role": "system", "content": system_preamble},
    {"role": "user", "content": user_content}
]

prompt_token_ids = tokenizer.apply_chat_template(
    conversation=messages,
    tokenize=True,
    add_generation_prompt=True 
)

# THE FIX: skip_special_tokens=True hides the ugly <BOS> and <|START_OF_TURN|> tags!
decoded_prompt_for_humans = tokenizer.decode(prompt_token_ids, skip_special_tokens=True)
token_count = len(prompt_token_ids)

print("="*80)
print(f"🛠️ DEBUG 2: CLEAN PROMPT (What the model 'reads' | Tokens: {token_count})")
print("-" * 80)
print(decoded_prompt_for_humans)
print("="*80)

# ==========================================
# 5. GENERATION
# ==========================================
sampling_params = SamplingParams(
    temperature=0.1, 
    max_tokens=400,
    repetition_penalty=1.05
)

outputs = llm.generate(
    prompts=[{"prompt_token_ids": prompt_token_ids}], 
    sampling_params=sampling_params, 
    use_tqdm=False
)

generated_text = outputs[0].outputs[0].text.replace("<|START_RESPONSE|>", "").strip()
latency = time.time() - start_time

# ==========================================
# 6. PIPELINE FINAL OUTPUT
# ==========================================
print("\n" + "="*80)
print(f"🤖 FINAL MODEL ANSWER (Latency: {latency:.2f}s | Docs Used: {accepted_count}):")
print("-" * 80)
print(generated_text)